# Local Miller equilibrium fitting

Fit several closed Miller surfaces, reconstruct their contours, inspect residuals, and demonstrate the deliberate near-separatrix rejection.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from vaft.data.equilibrium import Contour, MillerSurface
from vaft.process.equilibrium import evaluate_miller, fit_miller_surface

theta = np.linspace(0, 2*np.pi, 500, endpoint=False)
levels = (0.25, 0.50, 0.80, 0.999)
fits = []
for level in levels:
    surface = MillerSurface(r=0.35*np.sqrt(level), r0=1.0+0.03*level, z0=0.0, kappa=1.45+0.2*level, delta=0.25*level)
    r, z = evaluate_miller(surface, theta)
    fits.append(fit_miller_surface(Contour(r, z), radial_value=level))
    print(level, fits[-1].accepted, fits[-1].normalized_rms_error, fits[-1].reason)
assert all(item.accepted for item in fits[:-1])
assert not fits[-1].accepted

In [ ]:
fig, (ax, residual_ax) = plt.subplots(1, 2, figsize=(9, 4))
for item in fits:
    ax.plot(item.contour.r, item.contour.z, color="black", lw=1)
    ax.plot(item.reconstructed.r, item.reconstructed.z, "--", label=f"psi_n={item.surface.radial_value:g}")
residual_ax.plot(levels, [item.normalized_rms_error for item in fits], "o-")
residual_ax.axhline(0.02, color="tab:red", ls=":", label="fit tolerance")
ax.set_aspect("equal"); ax.legend(); residual_ax.legend()
ax.set(xlabel="R [m]", ylabel="Z [m]"); residual_ax.set(xlabel="psi_n", ylabel="normalized RMS")
plt.close(fig)